## **Predicting Loan Default Loss (Loss Given Default)**
# Business Context **bold text**
When a customer defaults on a loan, the lender does not necessarily lose the full outstanding balance. A portion of the exposure may be recovered through repayments, collateral liquidation, or legal recovery processes. However, the uncertainty around how much is ultimately recovered creates risk in capital planning, pricing, and portfolio management.

# **Problem Statement**
The organization currently faces the challenge of accurately estimating how much money is actually lost once a borrower has defaulted. Without reliable predictions, lenders may:

Hold excess regulatory capital
Under‑ or over‑price credit risk
Make suboptimal recovery and collection decisions

# **Proposed Solution**
This project proposes developing a Loss Given Default (LGD) prediction model that estimates the percentage of a loan exposure that is unrecoverable after a default has occurred. The model will use borrower characteristics, loan terms, credit history, and post‑default payment behavior to generate data‑driven LGD estimates.

# **Business Value**
Regulatory compliance: LGD is a core input in Basel III capital calculations, directly impacting risk‑weighted assets and capital requirements.

Capital efficiency: More accurate LGD estimates enable better alignment between true risk and capital held.

Improved risk management: Helps identify high‑loss segments and prioritize recovery strategies.

Better pricing and portfolio decisions: Supports risk‑based pricing and improves expected loss forecasting.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns


In [3]:
dataset = pd.read_csv("/content/accepted_2007_to_2018Q4.csv")

In [ ]:
dataset.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917368 entries, 0 to 917367
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 1.0+ GB


In [5]:
dataset.shape

(917368, 151)

In [6]:
list(dataset.columns)

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [7]:
dataset["loan_status"].value_counts()

,count
loan_status,
Fully Paid,419742
Current,371717
Charged Off,111321
Late (31-120 days),9238
In Grace Period,3505
Late (16-30 days),1817
Default,17
Charge,1


# **CREATING TARGET VARIABLE**

In [8]:
default_status = ["Charged Off", "Default", "Charge"]

In [9]:
df = dataset[dataset['loan_status'].isin(default_status)].copy()

In [10]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
13,66624733,NaN,18000.0,18000.0,18000.0,60 months,19.48,471.70,E,E2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
25,67849662,NaN,4225.0,4225.0,4225.0,36 months,14.85,146.16,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
30,67715283,NaN,16000.0,16000.0,16000.0,36 months,12.88,538.18,C,C2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
31,68341789,NaN,24250.0,24250.0,24250.0,60 months,24.24,701.01,F,F3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
33,68415473,NaN,25000.0,25000.0,25000.0,60 months,13.99,581.58,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df.shape

(111339, 151)

# **Compute LGD = LGD = 1 - (total_pymnt / funded_amnt)**
Total_pymnt includes: principal repaid + interest + late fees + recoveries
Funded_amnt is the actual amount disbursed to borrower


In [13]:
df['recovery_rate'] = df['total_pymnt'] / df['funded_amnt']
df['LGD'] = 1 - df['recovery_rate']

In [32]:
# Count Anomalies
n_neg = (df['LGD']<0).sum()
n_over = (df['LGD']>1).sum()

print(f" Over_recovery percentage: {n_neg} : ({(n_neg)/(len(df)*100):.2}%)")
print(f" Loss exceeds exposure: {n_over} : ({(n_over)/(len(df)*100):.2}%)")

 Over_recovery percentage: 6161 : (0.00055%)
 Loss exceeds exposure: 0 : (0.0%)


In [33]:
#Clean target
df['LGD_RAW'] = df['LGD'].copy()
df['LGD'] = df['LGD'].clip(0,1)

In [34]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,recovery_rate,LGD,LGD_RAW
13,66624733,NaN,18000.0,18000.0,18000.0,60 months,19.48,471.70,E,E2,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0.525152,0.474848,0.474848
25,67849662,NaN,4225.0,4225.0,4225.0,36 months,14.85,146.16,C,C5,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0.605650,0.394350,0.394350
30,67715283,NaN,16000.0,16000.0,16000.0,36 months,12.88,538.18,C,C2,...,N,NaN,NaN,NaN,NaN,NaN,NaN,1.087289,0.000000,-0.087289
31,68341789,NaN,24250.0,24250.0,24250.0,60 months,24.24,701.01,F,F3,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0.170079,0.829921,0.829921
33,68415473,NaN,25000.0,25000.0,25000.0,60 months,13.99,581.58,C,C4,...,N,NaN,NaN,NaN,NaN,NaN,NaN,0.579637,0.420363,0.420363


In [49]:
print(f"MEAN: {(df["LGD"].mean()):.2f})")
print(f"MEDIAN: {(df["LGD"].median()):.2f})")

MEAN: 0.48)
MEDIAN: 0.51)


In [51]:
leakage_cols = [
    # Payment outcomes (used to compute target)
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'out_prncp', 'out_prncp_inv',
    # Post-origination dates and status
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'loan_status',
    # Settlement info (post-default)
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    # Hardship program (post-default intervention)
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date',
    'hardship_end_date', 'payment_plan_start_date', 'hardship_length',
    'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    # Computed columns
    'recovery_rate', 'LGD_raw',
]

# --- IDENTIFIERS: Not predictive ---
id_cols = [
    'id', 'member_id', 'url', 'desc', 'emp_title', 'title',
    'pymnt_plan', 'policy_code',
]